# Colab launcher (thin)

Runs the same `diffusion_correction` package and `dlmc` commands as the Mac. No training logic lives here.
Checkpoints are written locally under `runs/` and each completed checkpoint is copied to Google Drive and re-hashed (`--durable-dir`). After a disconnect, rerun every cell: `--resume auto` picks up the local or Drive `latest` checkpoint and validates config, dataset and base-model identity.

No A100 run has been measured yet (D-030); bf16/TF32 are not enabled.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DURABLE = '/content/drive/MyDrive/dlmc'
!nvidia-smi

In [ ]:
%cd /content
!test -d Diffusion-Language-Model-Error-Correction || git clone https://github.com/ayujverma/Diffusion-Language-Model-Error-Correction.git
%cd /content/Diffusion-Language-Model-Error-Correction
!git rev-parse HEAD
!pip install -q -e '.[dev]'
!dlmc download-backbone
# Regenerate the pilot data; generate-data verifies it. The manifest hash must match the config.
!test -f runs/data/pilot-seed20261116/manifest.json || dlmc generate-data --config configs/data/pilot.yaml
!shasum -a 256 runs/data/pilot-seed20261116/manifest.json

In [ ]:
!dlmc doctor --backbone open-dcoder --device cuda
!dlmc benchmark --config configs/train/warmup_pilot.yaml --device cuda --configs 'full,32,mixed;full,32,longest' --warmup-steps 3 --timed-steps 10 --out runs/benchmark/pilot_cuda.json

In [ ]:
# Common warmup (or its continuation). Raise --max-updates only with a recorded reason.
!dlmc train --config configs/train/warmup_pilot.yaml --device cuda --resume auto --durable-dir {DURABLE}